# درس ۱۱ - پروتکل زمینهٔ مدل (MCP)

The **پروتکل زمینهٔ مدل (MCP)** is an open standard that enables agents to dynamically discover and use tools, resources, and data sources at runtime. Instead of hardcoding tools into an agent, MCP lets agents connect to external servers that expose capabilities on demand.

In this lesson, you'll learn:
- MCP چیست و چرا برای سیستم‌های مبتنی بر عامل اهمیت دارد
- معماری کلاینت-سرور MCP چگونه کار می‌کند
- نحوه ساخت عامل‌هایی که از مکانیسم کشف ابزار به سبک MCP استفاده می‌کنند


## راه‌اندازی

**پیش‌نیازها:**
- یک پروژه Azure AI Foundry با یک مدل مستقر
- اجرای `az login` برای احراز هویت


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## پروتکل زمینه مدل (MCP) چیست؟

MCP روشی استاندارد را برای کشف و تعامل عوامل هوش مصنوعی با ابزارها و منابع دادهٔ خارجی تعریف می‌کند:

- **MCP Server**: ابزارها، منابع و پرامپت‌ها را از طریق یک پروتکل استاندارد در دسترس قرار می‌دهد
- **MCP Client**: محیط اجرای عامل که به سرورها متصل می‌شود و قابلیت‌های در دسترس را کشف می‌کند
- **کشف پویا**: عوامل نیازی به ابزارهای سخت‌کدشده ندارند — آنچه در زمان اجرا در دسترس است را کشف می‌کنند

این موضوع برای ساخت سیستم‌های عامل قابل گسترش بسیار قدرتمند است، جایی که قابلیت‌های جدید می‌توانند بدون تغییر کد عامل اضافه شوند.


## نحوه کار MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. عامل (MCP client) به یک سرور MCP متصل می‌شود
2. سرور با فهرستی از ابزارهای در دسترس و طرحواره‌های آن‌ها پاسخ می‌دهد
3. سپس عامل می‌تواند در حین استدلال خود هر ابزار کشف‌شده‌ای را فراخوانی کند
4. نتایج از طریق همان پروتکل بازمی‌گردند


## شبیه‌سازی کشف ابزارهای MCP

از آنجا که یک سرور واقعی MCP نیاز به یک فرآیند سروری در حال اجرا دارد، الگو را با استفاده از توابع `@tool` نشان می‌دهیم؛ این توابع آنچه یک سرویس اقامتی متصل به MCP فراهم می‌آورد را شبیه‌سازی می‌کنند.

در محیط تولید، این ابزارها به‌صورت پویا از یک سرور MCP کشف می‌شوند، نه اینکه به‌صورت محلی تعریف شوند.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## ساخت یک عامل با ابزارهای سبک MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP در محیط تولید

در یک محیط تولید، MCP الگوهای قدرتمندی را امکان‌پذیر می‌سازد:

- **کشف پویا ابزارها**: عامل‌ها به سرورهای MCP متصل می‌شوند و در زمان اجرا ابزارها را کشف می‌کنند
- **معماری جداشده**: ارائه‌دهندگان ابزار می‌توانند به‌صورت مستقل از عامل به‌روزرسانی کنند
- **اشتراک‌گذاری بین‌سازمانی**: تیم‌ها می‌توانند قابلیت‌ها را از طریق سرورهای MCP عرضه کنند که هر عاملی می‌تواند از آن استفاده کند
- **پشتیبانی Microsoft Agent Framework**: MAF از طریق یکپارچه‌سازی `mcp` پشتیبانی کلاینت MCP را به‌صورت داخلی دارد

برای استفاده از یک سرور MCP واقعی با MAF، شما از طریق `hosted_mcp_tool()` یا یکپارچه‌سازی کلاینت MCP متصل می‌شوید.

**بیشتر بیاموزید:**
- [مشخصات MCP](https://modelcontextprotocol.io/)
- [پشتیبانی MCP در Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## خلاصه

در این درس آموختید:
- **MCP** یک استاندارد باز برای کشف پویأ ابزارها بین عامل‌ها و ارائه‌دهندگان ابزار است
- معماری **کلاینت-سرور** به عامل‌ها اجازه می‌دهد قابلیت‌ها را در زمان اجرا کشف کنند
- MCP امکان‌پذیر می‌سازد **سیستم‌های مبتنی بر عاملِ قابل توسعه و جداشده** که در آن‌ها می‌توان ابزارها را بدون تغییر کد اضافه کرد
- Microsoft Agent Framework پشتیبانی **MCP توکار** را برای استفاده در محیط تولید فراهم می‌کند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب مسئولیت:
این سند با استفاده از سرویس ترجمهٔ هوش مصنوعی Co-op Translator (https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی اشتباهات یا نادرستی‌هایی باشند. سند اصلی به زبان اصلی خود باید به‌عنوان مرجع معتبر تلقی شود. برای اطلاعات حساس یا حیاتی، ترجمهٔ حرفه‌ای توسط انسان توصیه می‌شود. ما در قبال هرگونه سوءتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
